# Modelado — ablación de clima

Comparamos base de seis predictores, clima observado, anomalías y ambos para h=2 y h=4. Las climatologías se ajustan por fold antes del primer origen de prueba. Las variantes se seleccionan solo con 2021–2023; 2024 y 2025 se consultan después.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.ablacion_clima import ejecutar_ablacion_clima

reporte = ejecutar_ablacion_clima()

h=2: 30095 filas comunes; 390 de arranque excluidas


  base_6: F1 validación reg=0.663, cls=0.594
  mas_clima_observado: F1 validación reg=0.646, cls=0.540
  mas_anomalias: F1 validación reg=0.642, cls=0.492
  mas_ambos: F1 validación reg=0.634, cls=0.490


  Elegidas h=2: {'regresion': 'base_6', 'clasificacion': 'base_6'}
h=4: 29965 filas comunes; 520 de arranque excluidas


  base_6: F1 validación reg=0.575, cls=0.555
  mas_clima_observado: F1 validación reg=0.556, cls=0.497
  mas_anomalias: F1 validación reg=0.547, cls=0.456
  mas_ambos: F1 validación reg=0.555, cls=0.470


  Elegidas h=4: {'regresion': 'base_6', 'clasificacion': 'base_6'}


In [2]:
import pandas as pd

filas = []
for h, bloque in reporte['horizontes'].items():
    for variante, resultado in bloque['variantes'].items():
        folds = resultado['folds_validacion']
        filas.append({'h': int(h), 'variante': variante, 'filas_comunes': bloque['filas_comparadas'],
                      'F1_regresion': sum(f['regresion']['alerta_derivada']['f1'] for f in folds) / len(folds),
                      'F1_clasificacion': sum(f['clasificacion']['alerta_directa']['f1'] for f in folds) / len(folds),
                      'elegida_regresion': variante == bloque['elegidas_solo_validacion']['regresion'],
                      'elegida_clasificacion': variante == bloque['elegidas_solo_validacion']['clasificacion']})
display(pd.DataFrame(filas).sort_values(['h', 'variante']))

,h,variante,filas_comunes,F1_regresion,F1_clasificacion,elegida_regresion,elegida_clasificacion
0,2,base_6,30095,0.663479,0.593724,True,True
3,2,mas_ambos,30095,0.633606,0.490364,False,False
2,2,mas_anomalias,30095,0.642060,0.492380,False,False
1,2,mas_clima_observado,30095,0.646451,0.539796,False,False
4,4,base_6,29965,0.575051,0.555106,True,True
7,4,mas_ambos,29965,0.555055,0.470396,False,False
6,4,mas_anomalias,29965,0.547253,0.456145,False,False
5,4,mas_clima_observado,29965,0.555712,0.497303,False,False


In [3]:
filas = []
for h, bloque in reporte['horizontes'].items():
    for variante, resultado in bloque['variantes'].items():
        for fold in resultado['folds_prueba']:
            for enfoque, metrica in (('regresion', fold['regresion']['alerta_derivada']),
                                     ('clasificacion', fold['clasificacion']['alerta_directa'])):
                filas.append({'h': int(h), 'bloque': fold['bloque'], 'variante': variante,
                              'enfoque': enfoque, 'positivos': metrica['positivos'],
                              'VP': metrica['vp'], 'FP': metrica['fp'], 'F1': metrica['f1']})
display(pd.DataFrame(filas).sort_values(['h', 'bloque', 'enfoque']))

,h,bloque,variante,enfoque,positivos,VP,FP,F1
3,2,calendario_2025,base_6,clasificacion,3,2,327,0.012048
2,2,calendario_2025,base_6,regresion,3,0,3,0.000000
1,2,temporada_2024,base_6,clasificacion,857,806,881,0.633648
0,2,temporada_2024,base_6,regresion,857,654,118,0.802947
7,4,calendario_2025,base_6,clasificacion,3,1,349,0.005666
6,4,calendario_2025,base_6,regresion,3,0,5,0.000000
5,4,temporada_2024,base_6,clasificacion,857,764,880,0.610956
4,4,temporada_2024,base_6,regresion,857,618,160,0.755963


El informe y las limitaciones están en `docs/modeling/ablacion_clima.md`. Los resultados de 2024 y 2025 no se utilizaron para seleccionar variables. La sociodemografía queda para una fase separada y su inicio necesita revisión de Rosa.